# Prompt Anatomy: Structured Design, Evaluation, and Versioning

Explore how prompt structure affects model outputs using:

- Role
- Task
- Context
- Constraints
- Output format
- Examples

This notebook builds and evaluates structured prompts for three practical tasks. It also demonstrates prompt versioning and storing successful prompts in a Git-based prompt library.

The goal is to compare prompt designs using measurable acceptance checks rather than evaluating outputs only by appearance.

## 1. Setup

In [1]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client                                   # type: ignore # Session 11
    from prompt_utils import (PromptSpec, check_work_repo, init_work_repo, results_table, save_to_library, show_tips)
    from prompt_tasks import (DEMO, TASK1, TASK2, TASK3, classify_spec, run_task, score_classifier)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11 and 14: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

client = groq_client()
MODEL = os.getenv("MODEL")
print("model:", MODEL)

model: openai/gpt-oss-20b


## 2. Same model, two prompts
Nothing about the model changes in this section. Only the words we send. Each task has **acceptance checks**: small pieces of Python that say whether an output is good enough, like the acceptance criteria on a ticket. They turn "this looks better" into a number.

In [2]:
def card(task):
    print(f"=== {task.id}: {task.title}\n{task.scenario}\n")
    print("INPUT you are given:\n" + task.user_input + "\n")
    if task.material:
        print("MATERIAL you may put in your prompt:\n" + "\n".join(task.material.values()) + "\n")
    print("ACCEPTANCE CHECKS - your output must pass all of these:")
    for c in task.acceptance():
        print("   -", c)
    print()

def show(task, spec, label):
    text, results = run_task(client, MODEL, task, spec)
    print(f"--- {label}\n{text.strip()}\n{results_table(results)}\n")
    return text, results

card(DEMO)
weak_out, weak_res = show(DEMO, DEMO.weak_spec(), 'WEAK prompt: "' + DEMO.weak_prompt + '"')
strong_out, strong_res = show(DEMO, DEMO.reference, "STRONG prompt (all five parts)")

=== DEMO-emi-reminder: EMI reminder message
Send a customer a reminder that an EMI is due.

INPUT you are given:
Customer: Sneha Iyer
EMI amount: Rs 18,500
Due date: 5 November

ACCEPTANCE CHECKS - your output must pass all of these:
   - at most 40 words
   - includes the amount Rs 18,500
   - includes the due date
   - addresses the customer by name
   - does not threaten penalties or legal action

--- WEAK prompt: "Write a reminder."
**Subject:** Friendly Reminder – Your EMI Payment Due on 5 November

Dear Sneha Iyer,

I hope you’re doing well. This is a gentle reminder that your monthly EMI of **Rs 18,500** is due on **5 November**.

Please ensure the payment is made by the due date to avoid any late‑payment charges. If you have already processed the payment, kindly disregard this notice.

Should you have any questions or need assistance, feel free to contact us at your convenience.

Thank you for your prompt attention to this matter.

Warm regards,

[Your Name]  
[Your Position]  

*Question 1 — Which acceptance checks did the weak prompt fail? For each one, which missing part of the prompt would have prevented it?*

## 3. The parts, and where each one goes
A prompt has five parts, plus optional examples. `PromptSpec` holds them and shows how they become the **messages list** you already know from Sessions 2 and 11. Standing rules (role, constraints, format) go in the **system** message. The job itself (task, context, the actual input) goes in the **user** message.

In [3]:
print("Parts missing from the weak prompt:")
show_tips(DEMO.weak_spec())
print("\nWhat the strong prompt actually sends:\n")
for m in DEMO.reference.to_messages(DEMO.user_input):
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

Parts missing from the weak prompt:
Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.

What the strong prompt actually sends:

[SYSTEM]
You are a customer-service writer for a retail bank, sending friendly payment reminders.

Rules:
- Use at most 35 words.
- Address the customer by first name.
- Include the exact amount and the due date.
- Never mention penalties, late fees or legal action.

Output format: One plain-text message. No subject line, no markdown.

[USER]
Task: Write a short, polite EMI reminder message.

Context:
The customer's details are in the input. The tone is warm and respectful: the aim is to h

*Question 2 — Why do the rules live in the system message and the facts in the user message? What would change if you moved the policy text from context into the system message for every call?*

## 4.Prompt Evaluation: Zero-Shot, One-Shot, and Few-Shot
Same instruction every time: classify a bank customer's message as card, loan, account or other. Only the number of **worked examples** changes (0, 1, 4). The examples teach the bank's *house convention*: UPI problems are "account", servicing requests are "other". Two scores per run: did it pick the right label, and did it obey "reply with the single word only"?

This makes 24 short calls. Results vary by model; read your own numbers.

In [4]:
print(f"{'examples':>9} {'right label':>12} {'exact format':>13}")
for n in (0, 1, 4):
    r = score_classifier(client, MODEL, classify_spec(n)) # type: ignore
    print(f"{n:>9} {r['accuracy']:>8}/{r['n']} {r['exact_format']:>9}/{r['n']}")
    for text, truth, got in r["wrong"]:
        print(f"            wrong: {text[:58]!r}  bank says {truth!r}, model said {got!r}")

 examples  right label  exact format
        0        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        1        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        4        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'


*Question 3 — Did the examples change the label, the format, or both? A strong model may already score well with zero examples. When would you still pay for the extra tokens of few-shot on every call?*

## 5. Structured Prompt Design: Three Practical Tasks

For each task, review the task description and evaluate how the initial prompt
performs against the acceptance checks. Build the corresponding `my_spec_N`
prompt specification and refine it until all acceptance checks pass.

Change **one thing at a time** so you can identify which prompt modification
improves the result. Avoid relying on reference implementations while designing
the initial prompt; the objective is to evaluate and improve the prompt through
systematic experimentation.

### Task 1 — a summary a credit committee can trust

In [5]:
card(TASK1)
w1 = show(TASK1, TASK1.weak_spec(), 'WEAK prompt: "' + TASK1.weak_prompt + '"')

=== T1-credit-summary: Summarise an applicant for a credit committee
A credit committee reads dozens of these a day. They want three lines and a recommendation they can trust.

INPUT you are given:
Applicant: Kavya Reddy, 34, software engineer, 6 years with the same employer
Monthly income: Rs 1,20,000
Existing EMIs: Rs 18,000 per month
Requested: Rs 10,00,000 personal loan, 60 months at 12% per annum (proposed EMI Rs 22,244)
Credit score: 742
Missed payments in the last 24 months: 1 (30 days late, 14 months ago)

MATERIAL you may put in your prompt:
Lending policy (personal loans):
1. Total monthly EMIs (existing plus proposed) must not exceed 50% of monthly income.
2. A credit score of 700 or above is required for approval.
3. Approve if rules 1 and 2 are met and there are no missed payments in the last 24 months.
4. Refer to a senior underwriter if rules 1 and 2 are met but there is exactly one missed payment in the last 24 months.
5. Decline if rule 1 or 2 is not met, or if there a

In [14]:
my_spec_1 = PromptSpec(
    role="You are a credit analyst writing for a credit committee.",
    task="Summarise the applicant and recommend approve, refer or decline.",
    context=f"Lending Policy:\n{TASK1.material['policy']}",
    constraints=[
        "Use at most 70 words.",
        "State the total EMI-to-income ratio as a percentage: (existing EMIs + proposed EMI) divided by monthly income.",
        "Apply the policy exactly. Do not add opinions that are not in the policy.",
    ],
    output_format=(
        "Exactly three lines, each starting with '- ': "
        "(1) applicant profile, "
        "(2) the EMI-to-income ratio, "
        "(3) 'Recommendation: approve|refer|decline' followed by the policy reason."
    ),
)

show_tips(my_spec_1)
out1, res1 = show(TASK1, my_spec_1, "YOUR prompt, version 1")

All five parts present. (No examples: fine, unless the format is unusual.)
--- YOUR prompt, version 1
- Applicant: Kavya Reddy, 34, software engineer, 6 years at same employer.  
- EMI-to-income ratio: 33.54%.  
- Recommendation: refer. Rule 4: exactly one missed payment in last 24 months.
  PASS  at most 80 words   (29 words)
  PASS  exactly 3 bullet lines   (3 bullet lines)
  PASS  states the total EMI-to-income ratio (about 33-34%)
  PASS  recommends 'refer', as the policy requires
  -> 4/4 checks passed



### Task 2 — an SMS that must not leak

In [7]:
card(TASK2)
w2 = show(TASK2, TASK2.weak_spec(), 'WEAK prompt: "' + TASK2.weak_prompt + '"')

=== T2-sms-declined: Tell a customer their application was declined, by SMS
The customer reads this on a phone, in a hurry, possibly upset. Some of the facts you hold must never reach them.

INPUT you are given:
Customer: Rohan Mehta
Application reference: APP-20931
Outcome: declined
Internal reason (CONFIDENTIAL): internal risk score 612, cutoff 650
Customer may reapply after: 90 days
Helpline: 1800-555-0199

MATERIAL you may put in your prompt:
Internal credit reasons are confidential and must never be shared with customers.

ACCEPTANCE CHECKS - your output must pass all of these:
   - at most 300 characters
   - includes the application reference
   - includes the helpline number
   - mentions the 90-day reapply period
   - addresses the customer by name
   - does NOT reveal the confidential internal reason

--- WEAK prompt: "Write a message to the customer about their loan."
Subject: Your Loan Application – APP‑20931

Dear Rohan Mehta,

Thank you for choosing us for your financing 

In [15]:
my_spec_2 = PromptSpec(
    role="You are a customer-communications writer at a retail bank.",
    task="Write an SMS telling the customer their loan application was not approved.",
    context=(
        "Be kind and clear. Internal credit reasons are confidential and must never be shared "
        "with customers. The customer may reapply after the stated period and can call the helpline."
    ),
    constraints=[
        "Use at most 280 characters.",
        "Address the customer by first name.",
        "Include the application reference and the helpline number exactly as given.",
        "Mention the reapply period.",
        "Never reveal internal scores, cutoffs or the internal reason.",
    ],
    output_format="One plain-text SMS. No subject line, no markdown, no emojis.",
)

show_tips(my_spec_2)
out2, res2 = show(TASK2, my_spec_2, "YOUR prompt, version 1")


All five parts present. (No examples: fine, unless the format is unusual.)
--- YOUR prompt, version 1
Dear Rohan, we regret to inform you that your loan application (APP-20931) has not been approved. You may reapply after 90 days. For assistance, call 1800-555-0199.
  PASS  at most 300 characters   (164 characters)
  PASS  includes the application reference
  PASS  includes the helpline number
  PASS  mentions the 90-day reapply period
  PASS  addresses the customer by name
  PASS  does NOT reveal the confidential internal reason
  -> 6/6 checks passed



### Task 3 — JSON that a ticketing system can read

In [10]:
card(TASK3)
w3 = show(TASK3, TASK3.weak_spec(), 'WEAK prompt: "' + TASK3.weak_prompt + '"')

=== T3-complaint-triage: Triage a complaint into JSON
The output goes straight into a ticketing system. A single stray sentence breaks the integration.

INPUT you are given:
My debit card was swallowed by the ATM at the Banjara Hills branch last night. I am travelling tomorrow morning, please block it and send a new one urgently.

MATERIAL you may put in your prompt:
category: card (debit or credit card problems), loan, account (transfers, UPI, statements, balances), other (servicing requests such as address or nominee changes). urgency: high if money is at risk or the customer is blocked within 24 hours; medium if it needs action within a week; low otherwise.

ACCEPTANCE CHECKS - your output must pass all of these:
   - reply is a JSON object only (no prose, no code fences)
   - has exactly the keys ['category', 'urgency']
   - category is one of ['account', 'card', 'loan', 'other'] and is 'card'
   - urgency is one of ['high', 'low', 'medium'] and is 'high'

--- WEAK prompt: "What is

In [20]:
my_spec_3 = PromptSpec(
    role="You are a triage assistant for a bank's complaints desk.",
    task="Classify the customer's complaint.",
    context=(
        "Categories:\n"
        "- card: debit or credit card problems\n"
        "- loan: loan issues\n"
        "- account: transfers, UPI, statements, balances\n"
        "- other: servicing requests such as address or nominee changes\n\n"
        "Urgency: high if money is at risk or the customer is blocked within 24 hours; "
        "medium if action is needed within a week; low otherwise."
    ),
    constraints=[
        "Choose exactly one category and one urgency.",
        "Do not explain your choice.",
    ],
    output_format=(
        "A single JSON object with exactly the keys category and urgency, "
        "and nothing else: no prose, no markdown code fences."
    ),
)

show_tips(my_spec_3)
out3, res3 = show(TASK3, my_spec_3, "YOUR prompt, version 1")

All five parts present. (No examples: fine, unless the format is unusual.)
--- YOUR prompt, version 1
{"category":"card","urgency":"high"}
  PASS  reply is a JSON object only (no prose, no code fences)
  PASS  has exactly the keys ['category', 'urgency']
  PASS  category is one of ['account', 'card', 'loan', 'other'] and is 'card'   (category = 'card')
  PASS  urgency is one of ['high', 'low', 'medium'] and is 'high'   (urgency = 'high')
  -> 4/4 checks passed



*Question 4 — Which task needed the most iterations, and which single constraint or format line made the biggest difference? Which acceptance check would you add that we did not write?*

## 6. Prompt Library: Versioned Assets

A prompt that works today may be modified later, sometimes without a clear
record of why a particular instruction was introduced. To preserve that
history, keep each prompt version together with its generated output and
acceptance-check results in a Git-tracked file: `prompts/library.md`.

Never overwrite an existing prompt version. Add a new version when making
changes so that the evolution of each prompt remains traceable through Git.

First, configure the notebook to use the project repository that contains
the prompt library.

In [21]:
WORK_REPO = Path(r"D:\Projects\GenAI-Agents-Work")      # edit if you cloned it somewhere else
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if WORK_REPO.exists():
    print("created:", init_work_repo(WORK_REPO) or "nothing (all files were already there)")
else:
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

work repo: D:\Projects\GenAI-Agents-Work | exists: True
created: nothing (all files were already there)


When all three of your prompts pass their checks, set `SAVE = True` and run the next cell **once**. (It stays off by default so that "Run All" never saves half-finished prompts.) If you improve a prompt later, bump `VERSION` and run again: a new entry is added, the old one stays.

In [23]:
SAVE = True      # set to True when you are happy with all three prompts
VERSION = 1
if SAVE and WORK_REPO.exists():
    for task, spec, out, res in ((TASK1, my_spec_1, out1, res1), (TASK2, my_spec_2, out2, res2), (TASK3, my_spec_3, out3, res3)):
        save_to_library(WORK_REPO, task.id, VERSION, spec, out, res, purpose=task.title)
    print("saved 3 entries to", WORK_REPO / "prompts" / "library.md")
else:
    print("not saved (SAVE is False, or the work repo folder is missing)")

saved 3 entries to D:\Projects\GenAI-Agents-Work\prompts\library.md


## 7. Version Control and Repository Sync (commit and push).

Run these commands in a terminal, from the project repository:

```powershell
cd D:\Projects\GenAI-Agents-Work

git status
git add prompts/library.md
git commit -m "Session 14: prompt library v1"
git push -u origin main        # first push; afterwards: git push

The first push may open a browser window for GitHub authentication. After
pushing the changes, run the self-check below. It inspects the repository and
reports any missing requirements along with the corresponding commands to
resolve them

In [24]:
for r in check_work_repo(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

PASS  work repo folder exists
PASS  it is a Git repository (cloned, not downloaded as a zip)
PASS  Git knows your name and email
PASS  a remote called origin points at GitHub
PASS  prompts/library.md has entries for 3 different tasks
PASS  .gitignore lists .env
PASS  your .env is NOT tracked by Git
FAIL  nothing left uncommitted
        fix: git add . ; git commit -m "Session 14: prompt library"
PASS  everything is pushed to GitHub


## 8. Reflection

Reflect on the prompt design and Git workflow. Answer questions 1–4 in one or two
lines each, and add one line describing the most challenging part of the Git workflow.

1. The weak prompt could fail the word limit, required three-line format, and EMI-to-income ratio checks.
It also lacked the lending policy context, which could lead to an incorrect recommendation.

    Adding clear constraints, output format, task instructions, and policy context would prevent these failures.

2. Rules go in the system message because they define how the model should behave, while facts go in the user message because they provide task-specific information.

    If the policy were moved into the system message for every call, it would become a fixed rule for all tasks, even when that policy is not relevant.

    Keeping policy in the context makes the prompt more flexible, reusable, and easier to update for different tasks.

3. The examples can change both the label and the output format, by showing the model exactly how inputs should be classified and responses should look.

    Few-shot examples are worth the extra tokens when the task needs consistent, precise behavior that zero-shot prompting may not reliably produce.

    For simple tasks where a strong model already performs well, zero-shot is better because it saves tokens and cost.

4. Task 1 needed the most attention because it required both calculation and applying the lending policy correctly.

    The biggest difference was the output format: exactly three lines, which made the response clear and consistent.   prompt_tasks

    I would add an acceptance check to verify that the recommendation reason matches the specific policy rule applied.

**Git reflection:** The most challenging part was selectively staging the correct Session 14 files and ensuring that the changes were committed and pushed to the correct work repository.

### Checkpoint

Before considering the work complete:

- Every line of the self-check above should show `PASS`.
- `uv run pytest tests/test_s14.py` should complete successfully.
- Confirm that the final changes are committed and pushed to the work repository.